# 10 — Modelo de notas com feedback explícito

**EP · Data Science Knowledge Base | Sistemas de Recomendação | Notebook 10/16**

**Execução:** Reinicie o kernel e execute as células em ordem. Observe as instruções específicas e as dependências opcionais indicadas neste notebook.


## Materiais complementares

- [Guia Técnico](reference_materials/10_feedback_explicito_modelo.pdf)
- [Cheat Sheet](cheat_sheets/cheat_sheet_10_feedback_explicito_modelo.pdf)

**Navegação:** os links são relativos à pasta deste notebook.

## Objetivos de aprendizagem
- Treinar preditor sem vazamento
- Aplicar vieses usuário/item com regularização

## Fundamentação conceitual
Um baseline para nota explícita é $\hat r_{ui}=\mu+b_u+b_i$. Estimamos vieses iterativamente sobre **pares observados de treinamento**. Para novos IDs, o viés é zero (fallback na média).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.metrics.pairwise import cosine_similarity
DATA = Path('datasets')
if not (DATA / 'ratings.csv').exists():
    raise FileNotFoundError('Abra o JupyterLab na pasta da Disciplina 09 (onde existe datasets/).')
ratings = pd.read_csv(DATA / 'ratings.csv')
movies = pd.read_csv(DATA / 'movies.csv')
print('Avaliações:', len(ratings), '| Filmes:', len(movies), '| Usuários:', ratings.userId.nunique())

Avaliações: 100836 | Filmes: 9742 | Usuários: 610


## Separação por interação

In [2]:
from sklearn.model_selection import train_test_split
train,test=train_test_split(ratings,test_size=.2,random_state=42)
mu=train.rating.mean(); reg=10.0
bu={};bi={}
for _ in range(8):
    bu=(train.assign(resid=lambda x:x.rating-mu-x.movieId.map(bi).fillna(0)).groupby('userId').resid.agg(['sum','count']).pipe(lambda x:x['sum']/(x['count']+reg))).to_dict()
    bi=(train.assign(resid=lambda x:x.rating-mu-x.userId.map(bu).fillna(0)).groupby('movieId').resid.agg(['sum','count']).pipe(lambda x:x['sum']/(x['count']+reg))).to_dict()
def predict(frame):return np.clip(mu+frame.userId.map(bu).fillna(0)+frame.movieId.map(bi).fillna(0),.5,5)
print('Exemplo:',predict(test.head()))

Exemplo: 67037    3.224653
42175    3.340264
93850    2.558045
6187     3.751334
12229    3.438749
dtype: float64


**Interpretação.** Somente os dados de treino entram nos cálculos.

## Medição de erro

In [3]:
from sklearn.metrics import mean_squared_error,mean_absolute_error
p=predict(test);print('MAE',round(mean_absolute_error(test.rating,p),3),'RMSE',round(float(np.sqrt(mean_squared_error(test.rating,p))),3))

MAE 0.676 RMSE 0.882


**Interpretação.** MAE/RMSE são métricas de predição, não necessariamente de utilidade Top-K.

## Aprofundamento e aplicação

**Ponto de atenção deste capítulo:** baseline e notas. Este complemento não substitui as seções e exemplos anteriores.

**Verificação conceitual.** Explique, com suas palavras, em qual etapa do fluxo de trabalho o tema **baseline e notas** pode produzir conclusões enganosas. Indique um indicador observável, um teste mínimo e uma limitação do procedimento.

**Exercício de extensão.** Reproduza o experimento central deste notebook alterando **um parâmetro por vez**. Compare com uma linha de base simples, reporte a variação da métrica pertinente e justifique o resultado. Quando houver treino/teste, qualquer etapa que aprenda parâmetros deve usar apenas o treino.

**Checklist de conclusão:** (1) entradas identificadas; (2) sementes registradas; (3) saídas interpretadas; (4) premissas explicitadas; (5) dados originais não sobrescritos.


### Exercício resolvido — protocolo mínimo de validação

**Enunciado.** Proponha uma comparação para **baseline e notas** sem contaminar a avaliação.

**Solução orientadora.** Defina uma hipótese falsificável, escolha um baseline simples, fixe uma divisão adequada ao problema, altere uma única decisão metodológica e compare métricas no conjunto reservado. Não transforme exemplos sintéticos em conclusões sobre populações reais. Documente a configuração e os casos de falha. Quando o tema não envolver predição, substitua o baseline por uma implementação de referência e compare correção, cobertura e custo.


## Análise crítica — Vieses e avaliação prospectiva

O baseline `r̂_ui = μ + b_u + b_i` utiliza média e vieses estimados exclusivamente no treino. Para usuário ou item novo, o componente desconhecido é zero e permanece a média acrescida de qualquer viés conhecido. Em produção, o clipping deve respeitar a escala válida; **divisão aleatória por evento não simula previsão futura**.

**Origem:** aprofundamento técnico autoral fundamentado no experimento deste notebook. Não é transcrição dos materiais históricos.

In [4]:
# Comparar modelo de vieses com baseline constante usando o MESMO teste
from sklearn.metrics import mean_absolute_error
pred_base = np.full(len(test), mu)
pred_vieses = predict(test)
print('MAE média global:', round(float(mean_absolute_error(test.rating,pred_base)),3))
print('MAE com vieses:', round(float(mean_absolute_error(test.rating,pred_vieses)),3))

MAE média global: 0.832
MAE com vieses: 0.676


### Critério de leitura

Interprete os resultados como diagnóstico didático. Para comparar modelos de recomendação, defina previamente população, itens elegíveis, janela temporal, sinal de relevância, baseline e métrica; mantenha treino, validação e teste separados.

**Atividade:** registre o efeito de uma mudança de parâmetro e explique uma ameaça à validade do experimento.

<!-- EP_FASE2_AREA09_LOTE2 -->
### Avaliação complementar — vieses com corte temporal global

O modelo anterior é ajustado em uma divisão **aleatória por interação**. Para verificar o impacto da ordem cronológica, construímos uma divisão temporal global pelo quantil 80% dos timestamps. Reajustamos a **média e os vieses de usuário e item exclusivamente no passado**; comparamos, sobre as mesmas notas posteriores, o baseline da média do treino e o modelo de vieses.

**Interpretação e limites:** o histórico até o corte é tratado como disponível antes do período de teste. Para usuários ou filmes ausentes no treino, o viés correspondente é zero; informamos a cobertura de pares conhecidos. O corte por quantil é didático, não define uma data de implantação, e não há ajuste de hiperparâmetros por validação. MAE/RMSE sobre notas **observadas** não mensuram qualidade de ranking ou exposição de itens não avaliados.


In [5]:
# Reajuste temporal completo; não reutiliza parâmetros da divisão aleatória acima.
from sklearn.metrics import mean_absolute_error, mean_squared_error
ep10_corte = ratings.timestamp.quantile(0.80)
ep10_train = ratings.loc[ratings.timestamp <= ep10_corte].copy()
ep10_test = ratings.loc[ratings.timestamp > ep10_corte].copy()
assert len(ep10_train) > 0 and len(ep10_test) > 0
ep10_mu = float(ep10_train.rating.mean())
ep10_reg = 10.0
ep10_bu, ep10_bi = {}, {}
for _ in range(8):
    ep10_bu = (ep10_train.assign(
        resid=lambda x: x.rating - ep10_mu - x.movieId.map(ep10_bi).fillna(0)
    ).groupby('userId').resid.agg(['sum', 'count']).pipe(
        lambda x: x['sum'] / (x['count'] + ep10_reg)
    )).to_dict()
    ep10_bi = (ep10_train.assign(
        resid=lambda x: x.rating - ep10_mu - x.userId.map(ep10_bu).fillna(0)
    ).groupby('movieId').resid.agg(['sum', 'count']).pipe(
        lambda x: x['sum'] / (x['count'] + ep10_reg)
    )).to_dict()
ep10_pred_media = np.full(len(ep10_test), ep10_mu)
ep10_pred_vieses = np.clip(
    ep10_mu + ep10_test.userId.map(ep10_bu).fillna(0).to_numpy()
    + ep10_test.movieId.map(ep10_bi).fillna(0).to_numpy(), 0.5, 5.0
)
ep10_y = ep10_test.rating.to_numpy(dtype=float)
ep10_usuario_conhecido = ep10_test.userId.isin(ep10_train.userId).to_numpy()
ep10_item_conhecido = ep10_test.movieId.isin(ep10_train.movieId).to_numpy()
ep10_cobertura = ep10_usuario_conhecido & ep10_item_conhecido
for ep10_nome, ep10_pred in [('media_passado', ep10_pred_media),
                             ('vieses_passado', ep10_pred_vieses)]:
    ep10_mae = float(mean_absolute_error(ep10_y, ep10_pred))
    ep10_rmse = float(np.sqrt(mean_squared_error(ep10_y, ep10_pred)))
    print(f'{ep10_nome}: MAE={ep10_mae:.4f}; RMSE={ep10_rmse:.4f}')
print(f'Treino temporal: {len(ep10_train)} | Teste: {len(ep10_test)}')
print(f'Pares com usuário E item conhecidos no treino: {ep10_cobertura.mean():.1%}')
for ep10_nome, ep10_mask in [
    ('ambos_conhecidos', ep10_cobertura),
    ('usuario_novo_item_conhecido', ~ep10_usuario_conhecido & ep10_item_conhecido),
    ('usuario_conhecido_item_novo', ep10_usuario_conhecido & ~ep10_item_conhecido),
    ('ambos_novos', ~ep10_usuario_conhecido & ~ep10_item_conhecido),
]:
    if ep10_mask.any():
        ep10_mae_segm = float(mean_absolute_error(ep10_y[ep10_mask], ep10_pred_vieses[ep10_mask]))
        print(f'Segmento {ep10_nome}: n={ep10_mask.sum()} | MAE vieses={ep10_mae_segm:.4f}')
    else:
        print(f'Segmento {ep10_nome}: sem exemplos')
assert np.isfinite(ep10_pred_vieses).all()


media_passado: MAE=0.8540; RMSE=1.0787
vieses_passado: MAE=0.7862; RMSE=1.0160
Treino temporal: 80669 | Teste: 20167
Pares com usuário E item conhecidos no treino: 6.5%
Segmento ambos_conhecidos: n=1310 | MAE vieses=0.6750
Segmento usuario_novo_item_conhecido: n=15813 | MAE vieses=0.7772
Segmento usuario_conhecido_item_novo: n=610 | MAE vieses=0.8187
Segmento ambos_novos: n=2434 | MAE vieses=0.8963


## Limitações e boas práticas
Divisão aleatória serve à demonstração, mas não substitui uma validação temporal em aplicação real.

## Exercícios
Compare com o preditor constante `mu` sem mudar a partição.

## Soluções comentadas
`mean_absolute_error(test.rating,np.full(len(test),mu))` calcula o baseline constante.

## Síntese
Reveja os resultados e compare métodos usando um protocolo de dados e métricas consistente. Não trate avaliações ausentes como avaliações negativas.

### Exportação de resultados

A próxima célula grava os resultados do capítulo em `outputs/`.


In [6]:
# Exportação de resultados do capítulo
# Execute todas as células em sequência antes de exportar.
# Cada arquivo documenta um experimento ou protocolo distinto.
from pathlib import Path
out = Path('outputs')
out.mkdir(parents=True, exist_ok=True)
pd.DataFrame({"userId": test.userId.to_numpy(), "movieId": test.movieId.to_numpy(),
              "nota_real": test.rating.to_numpy(), "predito_vieses": predict(test),
              "predito_media_global": np.full(len(test), mu)}).to_csv(
    out / "10_predicoes_teste.csv", index=False)

# Resultados das análises metodológicas complementares
from sklearn.metrics import mean_absolute_error, mean_squared_error
_ep_segmentos = [
    ('total', np.ones(len(ep10_y), dtype=bool)),
    ('ambos_conhecidos', ep10_cobertura),
    ('usuario_novo_item_conhecido', ~ep10_usuario_conhecido & ep10_item_conhecido),
    ('usuario_conhecido_item_novo', ep10_usuario_conhecido & ~ep10_item_conhecido),
    ('ambos_novos', ~ep10_usuario_conhecido & ~ep10_item_conhecido),
]
_ep_linhas = []
for _ep_modelo, _ep_pred in [('media_passado', ep10_pred_media),
                             ('vieses_passado', ep10_pred_vieses)]:
    for _ep_segmento, _ep_mask in _ep_segmentos:
        _ep_n = int(_ep_mask.sum())
        _ep_linhas.append({
            'modelo': _ep_modelo, 'segmento': _ep_segmento, 'n_teste': _ep_n,
            'mae': float(mean_absolute_error(ep10_y[_ep_mask], _ep_pred[_ep_mask])) if _ep_n else None,
            'rmse': float(np.sqrt(mean_squared_error(ep10_y[_ep_mask], _ep_pred[_ep_mask]))) if _ep_n else None,
            'n_treino_total': len(ep10_train), 'n_teste_total': len(ep10_test),
            'cobertura_pares_warm': float(ep10_cobertura.mean()),
            'protocolo': 'corte_global_timestamp_80pct_treino_apenas_passado'
        })
_ep_destino = out / '10_metricas_temporais_segmentadas.csv'
pd.DataFrame(_ep_linhas).to_csv(_ep_destino, index=False)
print('Arquivo exportado:', _ep_destino)


Arquivo exportado: outputs/10_metricas_temporais_segmentadas.csv
